# Table of Contents
- [Description](#description)
- [Read in data file](#read-in-data-file)
- [Mapping OCC 10 to O*NET-SOC 2019](#mapping-occ10-to-onet-soc-2019)
- [Manual Mapping](#manual-mapping)

# Description
The purpose of this notebook is to construct our GSS-based dataset, primarily by: 
1. Taking an initial look at the variables that the dataset offers,
2. Renaming them to be more readable,
3. Mapping OCC10 (Census 2010) codes to O*NET-SOC 2019

And then saving the resulting DataFrame off to a CSV under the `/data/constructed_datasets/` folder.

## Read in data file

In [1]:
import pandas as pd
import pyreadstat

data_dir = "../../data/"
spss_dir = data_dir + "spss_data/"
path = spss_dir + "GSS_job.sav"

_, meta = pyreadstat.read_sav(path, metadataonly=True)

In [3]:
col_data = list(zip(meta.column_names, meta.column_labels))
col_data

[('ETHNIC', 'country of family origin'),
 ('BALLOT', 'ballot used for interview'),
 ('WRKSTAT', 'labor force status'),
 ('OCC10', "r's census occupation code (2010)"),
 ('MARITAL', 'marital status'),
 ('SIBS', 'number of brothers and sisters'),
 ('AGE', 'age of respondent'),
 ('EDUC', 'highest year of school completed'),
 ('SEX', 'respondents sex'),
 ('RACE', 'race of respondent'),
 ('INCOME', 'total family income'),
 ('RELIG16', 'religion in which raised'),
 ('SATJOB', 'work satisfaction'),
 ('JOBMEANS', 'work important and feel accomplishment'),
 ('JOBKEEP', 'job security'),
 ('JOBRISE', 'opportunity for advancement'),
 ('JOBHONOR', 'occupation is recognized and respected'),
 ('JOBOFF', 'leaves time for leisure'),
 ('JOBINTER', 'interesting work'),
 ('JOBINDEP', 'allows one to work independently'),
 ('JOBRESP', 'responsible job tasks'),
 ('JOBPEOP', 'contact with other people'),
 ('JOBHELP', 'can help others'),
 ('JOBSOC', 'useful to society'),
 ('JOBACCMP', 'doing something meaningf

In [4]:
df, meta = pyreadstat.read_sav(path, user_missing=False)
df.head()

,ETHNIC,BALLOT,WRKSTAT,OCC10,MARITAL,SIBS,AGE,EDUC,SEX,RACE,...,EXJOBSAT,NEWSKILL,LOWPAY,TEMPWORK,MORTRAVL,ECONSUP1,WORECSUP,SATJOBHV,ID_,YEAR
0,NaN,NaN,1.0,520.0,5.0,3.0,23.0,16.0,2.0,1.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.0,1972.0
1,14.0,NaN,5.0,7700.0,1.0,4.0,70.0,10.0,1.0,1.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2.0,1972.0
2,15.0,NaN,2.0,4920.0,1.0,5.0,48.0,12.0,2.0,1.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,3.0,1972.0
3,37.0,NaN,1.0,800.0,1.0,5.0,27.0,17.0,2.0,1.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,4.0,1972.0
4,27.0,NaN,7.0,5020.0,1.0,2.0,61.0,12.0,2.0,1.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,5.0,1972.0


In [5]:
df["OCC10"].isna().sum()

np.int64(5337)

In [6]:
list(df.columns)

['ETHNIC',
 'BALLOT',
 'WRKSTAT',
 'OCC10',
 'MARITAL',
 'SIBS',
 'AGE',
 'EDUC',
 'SEX',
 'RACE',
 'INCOME',
 'RELIG16',
 'SATJOB',
 'JOBMEANS',
 'JOBKEEP',
 'JOBRISE',
 'JOBHONOR',
 'JOBOFF',
 'JOBINTER',
 'JOBINDEP',
 'JOBRESP',
 'JOBPEOP',
 'JOBHELP',
 'JOBSOC',
 'JOBACCMP',
 'JOBSAFE',
 'IMPPROMO',
 'ILIKEJOB',
 'TIMEPDWK',
 'WRKEARN',
 'WRKENJOY',
 'SECJOB',
 'HIINC',
 'PROMOTN',
 'LEISURE',
 'INTJOB',
 'WRKINDP',
 'HLPOTHS',
 'HLPSOC',
 'FLEXHRS',
 'EMPSELF',
 'SMALLBIG',
 'PRIVGOVT',
 'HRSMONEY',
 'JOBSAT',
 'WANTJOB1',
 'FLEXHRS1',
 'WKPERSNL',
 'FAMORJOB',
 'JOBORFAM',
 'EXJOBSAT',
 'NEWSKILL',
 'LOWPAY',
 'TEMPWORK',
 'MORTRAVL',
 'ECONSUP1',
 'WORECSUP',
 'SATJOBHV',
 'ID_',
 'YEAR']

## Mapping OCC10 to O*NET-SOC 2019

In [17]:
df_census_onet = pd.read_csv(data_dir + "crosswalks/census2010_to_onet2019.csv")
df_census_onet.drop(columns=["2010 Census Title", 
                             "O*NET-SOC 2019 Title"], 
                  inplace=True)

df_census_onet.head()

,2010 Census Code,O*NET-SOC 2019 Code
0,10,11-1011.00
1,20,11-1021.00
2,30,11-1031.00
3,40,11-2011.00
4,50,11-2022.00


In [18]:
census_to_onet = {census:onet for census,onet 
                  in zip(df_census_onet["2010 Census Code"],
                         df_census_onet["O*NET-SOC 2019 Code"])}

df["ONET_SOC_CODE"] = df["OCC10"].map(census_to_onet)

In [19]:
df.head()

,ETHNIC,BALLOT,WRKSTAT,OCC10,MARITAL,SIBS,AGE,EDUC,SEX,RACE,...,NEWSKILL,LOWPAY,TEMPWORK,MORTRAVL,ECONSUP1,WORECSUP,SATJOBHV,ID_,YEAR,ONET_SOC_CODE
0,NaN,NaN,1.0,520.0,5.0,3.0,23.0,16.0,2.0,1.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.0,1972.0,13-1022.00
1,14.0,NaN,5.0,7700.0,1.0,4.0,70.0,10.0,1.0,1.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2.0,1972.0,51-1011.00
2,15.0,NaN,2.0,4920.0,1.0,5.0,48.0,12.0,2.0,1.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,3.0,1972.0,41-9022.00
3,37.0,NaN,1.0,800.0,1.0,5.0,27.0,17.0,2.0,1.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,4.0,1972.0,13-2011.00
4,27.0,NaN,7.0,5020.0,1.0,2.0,61.0,12.0,2.0,1.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,5.0,1972.0,43-2021.00


In [20]:
missing_onet = df["ONET_SOC_CODE"].isna()
valid_census = df["OCC10"].notna()
df_need_mapping = df[missing_onet & valid_census]

In [21]:
df_need_mapping

,ETHNIC,BALLOT,WRKSTAT,OCC10,MARITAL,SIBS,AGE,EDUC,SEX,RACE,...,NEWSKILL,LOWPAY,TEMPWORK,MORTRAVL,ECONSUP1,WORECSUP,SATJOBHV,ID_,YEAR,ONET_SOC_CODE
32,26.0,NaN,4.0,9830.0,5.0,1.0,26.0,16.0,1.0,1.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,33.0,1972.0,NaN
60,11.0,NaN,4.0,9830.0,1.0,2.0,39.0,12.0,1.0,1.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,61.0,1972.0,NaN
90,26.0,NaN,4.0,9820.0,1.0,3.0,25.0,16.0,1.0,1.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,91.0,1972.0,NaN
148,8.0,NaN,1.0,9830.0,1.0,4.0,38.0,10.0,1.0,1.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,149.0,1972.0,NaN
179,NaN,NaN,7.0,9830.0,5.0,5.0,21.0,12.0,2.0,1.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,180.0,1972.0,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
74534,NaN,3.0,1.0,9830.0,1.0,4.0,25.0,17.0,2.0,2.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2145.0,2024.0,NaN
74535,999.0,3.0,1.0,9830.0,1.0,3.0,32.0,15.0,1.0,3.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2146.0,2024.0,NaN
74788,7.0,2.0,5.0,9830.0,2.0,1.0,54.0,18.0,2.0,1.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2399.0,2024.0,NaN
74790,11.0,1.0,1.0,9830.0,5.0,4.0,28.0,16.0,1.0,1.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2401.0,2024.0,NaN


In [23]:
print(f"The following codes need an O*NET-SOC mapping: {df_need_mapping["OCC10"].unique()}")

The following codes need an O*NET-SOC mapping: [9830. 9820. 9800. 9810.]


## Manual mapping
Using this as a source to compare to O*NET-SOC 2019 codes: 
- https://usa.ipums.org/usa-action/variables/occ2010#codes_section

I tried to match what seemed to fit the job title the most, but note that there is no direct path between the Census codes below and a specific O*NET-SOC 2019 code.

Census code 9830 was too ambiguous (Military, Rank Not Specified), so I have left it as blank (treat as N/A) for now.

In [31]:
# Military Officer Special and Tactical Operations Leaders
census_to_onet[9800] = "55-1019.00"

# First-Line Supervisors of Weapons Specialists/Crew Members
census_to_onet[9810] = "55-2012.00"

# Military Enlisted Tactical Operations and Air/Weapons Specialists and Crew Members
census_to_onet[9820] = "55-3019.00"

# This is very ambiguous and has no job title that is similar, so leave blank for now...
# census_to_onet[9830] = 

In [32]:
df["ONET_SOC_CODE"] = df["OCC10"].map(census_to_onet)

In [33]:
missing_onet = df["ONET_SOC_CODE"].isna()
valid_census = df["OCC10"].notna()
df_need_mapping = df[missing_onet & valid_census]

df_need_mapping.shape

(279, 61)

In [34]:
# Write to CSV
output_path = data_dir + "constructed_datasets/gss.csv"
df.to_csv(output_path, index=False)